In [2]:
import sys, os

sys.path.insert(0, os.path.join(os.getcwd() + "/" + "src"))
from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration
import torch

In [3]:
modelPath = "../models/rag-sequence-nq"
model: RagSequenceForGeneration = RagSequenceForGeneration.from_pretrained(modelPath)

/home/huahua/miniconda3/envs/transformers4rag/lib/python3.8/site-packages/transformers/modeling_utils.py:927: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torc

In [4]:
def iteration(model, level=0):
    for name, module in model.named_children():
        print(" " * level + name)
        iteration(module, level + 1)

In [5]:
iteration(model)

rag
 question_encoder
  question_encoder
   bert_model
    embeddings
     word_embeddings
     position_embeddings
     token_type_embeddings
     LayerNorm
     dropout
    encoder
     layer
      0
       attention
        self
         query
         key
         value
         dropout
        output
         dense
         LayerNorm
         dropout
       intermediate
        dense
       output
        dense
        LayerNorm
        dropout
      1
       attention
        self
         query
         key
         value
         dropout
        output
         dense
         LayerNorm
         dropout
       intermediate
        dense
       output
        dense
        LayerNorm
        dropout
      2
       attention
        self
         query
         key
         value
         dropout
        output
         dense
         LayerNorm
         dropout
       intermediate
        dense
       output
        dense
        LayerNorm
        dropout
      3
       attention
 

In [10]:
type(model.generator)

transformers.modeling_bart.BartForConditionalGeneration

In [23]:
scriptModel = torch.jit.trace(model.generator.model, torch.rand(10 * 10).to(torch.int).view(10, 10), strict=False)

/home/huahua/miniconda3/envs/transformers4rag/lib/python3.8/site-packages/transformers/modeling_bart.py:217: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not padding_mask.any():
/home/huahua/miniconda3/envs/transformers4rag/lib/python3.8/site-packages/transformers/modeling_bart.py:661: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert embed_dim == self.embed_dim
/home/huahua/miniconda3/envs/transformers4rag/lib/python3.8/site-packages/transformers/modeling_bart.py:662: TracerWarning: Converting a tensor to a Python boolean might cause the trace 

In [24]:
print(scriptModel.code)

def forward(self,
    input_ids: Tensor) -> Dict[str, Tensor]:
  decoder = self.decoder
  decoder0 = self.decoder
  embed_tokens = decoder0.embed_tokens
  encoder = self.encoder
  shared = self.shared
  weight = shared.weight
  prev_output_tokens = torch.clone(input_ids)
  _0 = torch.sum(torch.ne(input_ids, 1), [1])
  index_of_eos = torch.unsqueeze(torch.sub(_0, CONSTANTS.c0), -1)
  _1 = torch.gather(input_ids, 1, index_of_eos)
  _2 = torch.squeeze(_1)
  _3 = torch.slice(prev_output_tokens, 0, 0, 9223372036854775807)
  _4 = torch.copy_(torch.select(_3, 1, 0), _2)
  _5 = torch.slice(input_ids, 0, 0, 9223372036854775807)
  _6 = torch.slice(_5, 1, 0, -1)
  _7 = torch.slice(prev_output_tokens, 0, 0, 9223372036854775807)
  _8 = torch.slice(_7, 1, 1, 9223372036854775807)
  _9 = torch.copy_(_8, _6)
  tgt_len = ops.prim.NumToTensor(torch.size(prev_output_tokens, 1))
  _10 = int(tgt_len)
  t = torch.zeros([int(tgt_len), _10], dtype=None, layout=None, device=torch.device("cpu"), pin_memory=False